# Non-uniform Conditional Generation
Rebuild the non-uniform conditional-generation figure directly from the trained checkpoint, while handling the architecture mismatch between the stored config and some checkpoints.

In [ ]:
from pathlib import Path
import sys
import yaml

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import Config
from src.model.flow_matching_model.samples import Samples
import src.model.flow_matching_model as flow_matching

NOTEBOOK_DIR = ROOT / 'figures' / 'paper' / 'non_uniform_conditioning'
FIGURE_PATH = NOTEBOOK_DIR / 'non_uniform_conditional_generation.png'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 5
SIGMA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.frameon': False,
})


In [ ]:
def build_non_uniform_config(checkpoint_name: str) -> Config:
    # NOTE: the transformer model path has been removed from the codebase (only
    # UNet-based checkpoints are supported now), so this no longer needs to branch
    # on the checkpoint's architecture.
    config_path = ROOT / 'outputs' / 'gaussian prior - conditional generation non uniform - new visualization' / 'config.yaml'
    with open(config_path, 'r') as handle:
        base_config = yaml.safe_load(handle)
    return Config(**base_config)


def load_non_uniform_model(checkpoint_name: str):
    config = build_non_uniform_config(checkpoint_name)
    wrapper_cls = getattr(flow_matching, config.lightning_wrapper)
    model = wrapper_cls.load_from_checkpoint(
        ROOT / 'outputs' / 'gaussian prior - conditional generation non uniform - new visualization' / checkpoint_name,
        config=config,
        map_location=DEVICE,
    )
    return model.eval().to(DEVICE), config


def absolute_to_relative_time(curves: torch.Tensor) -> torch.Tensor:
    values = curves[:, 0]
    times = curves[:, 1]
    relative_times = torch.nn.functional.pad(times[:, 1:] - times[:, :-1], [1, 0, 0, 0])
    return torch.stack([values, relative_times], dim=1)


def relative_to_absolute_time(curves: torch.Tensor) -> torch.Tensor:
    values = curves[:, 0]
    times = curves[:, 1].cumsum(dim=-1)
    return torch.stack([values, times], dim=1)


def log_likelihood(curves, curve_times, lambdas, sigmas):
    t = curve_times[:, :-1]
    t_next = curve_times[:, 1:]
    x_t = curves[:, :-1]
    x_t_next = curves[:, 1:]
    delta_t = t_next - t
    means = x_t * (lambdas[:, None] ** delta_t)
    variances = sigmas[:, None] ** 2 * ((1 - (lambdas[:, None] ** (2 * delta_t))) / (1 - (lambdas[:, None] ** 2)))
    return ((-0.5 * torch.log(2 * torch.pi * variances)) - ((x_t_next - means) ** 2) / (2 * variances)).sum(dim=-1)


def estimate_coeffs(curves, curve_times, steps: int = 120, lr: float = 0.05):
    lambdas = torch.full((curves.shape[0],), 0.5, device=curves.device)
    sigmas = torch.full_like(lambdas, SIGMA)
    lambdas.requires_grad_(True)
    optimizer = torch.optim.Adam([lambdas], lr=lr)
    for _ in range(steps):
        with torch.enable_grad():
            loss = -log_likelihood(curves, curve_times, lambdas, sigmas).sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            lambdas.clamp_(0.01, 0.95)
    return lambdas.detach()


In [ ]:

CHECKPOINT_NAME = 'step=step=025000-v1.ckpt'
TARGET_COEFFS = [0.2, 0.4, 0.6, 0.8]
BIN_HALF_WIDTH = 0.03
NUM_CONDITIONING_CURVES = 36
NUM_GENERATED_PER_BIN = 36

model, model_config = load_non_uniform_model(CHECKPOINT_NAME)
dataset_curves = torch.tensor(np.load(ROOT / 'datasets' / 'ar1_non_uniform_15000_days' / 'curves.npy'), dtype=torch.float32)
dataset_coeffs = torch.tensor(np.load(ROOT / 'datasets' / 'ar1_non_uniform_15000_days' / 'coeffs.npy'), dtype=torch.float32)

results = {}
example_payloads = {}
for target in TARGET_COEFFS:
    mask = (dataset_coeffs > target - BIN_HALF_WIDTH) & (dataset_coeffs < target + BIN_HALF_WIDTH)
    conditioning_curves = dataset_curves[mask][:NUM_CONDITIONING_CURVES].to(DEVICE)
    relative_conditioning_curves = absolute_to_relative_time(conditioning_curves)
    display_conditioning_curves = conditioning_curves

    with torch.no_grad():
        encoded = model.encode(model.to_domain(Samples(relative_conditioning_curves[:, :, :256], domain='time', t=1)).curves)
        generated = model.domain_sample(NUM_GENERATED_PER_BIN, condition=encoded[:NUM_GENERATED_PER_BIN], num_steps=60).curves
        fixed_generated = relative_to_absolute_time(generated)

    estimated_coeffs = estimate_coeffs(fixed_generated[:, 0], fixed_generated[:, 1]).cpu()
    results[target] = {
        'estimated_coeffs': estimated_coeffs,
        'conditioning_coeffs': dataset_coeffs[mask][:NUM_CONDITIONING_CURVES].cpu(),
    }
    example_payloads[target] = {
        'conditioning_curve': display_conditioning_curves[0].detach().cpu(),
        'generated_curves': fixed_generated[:3].detach().cpu(),
    }

summary = {
    target: round(float(payload['estimated_coeffs'].mean()), 4)
    for target, payload in results.items()
}
summary


In [ ]:
fig = plt.figure(figsize=(12.0, 5.0))
grid = fig.add_gridspec(2, 4, width_ratios=[1.4, 1, 1, 1], height_ratios=[1, 1])

ax_summary = fig.add_subplot(grid[:, 0])
positions = np.arange(len(TARGET_COEFFS))
model_means = [results[target]['estimated_coeffs'].mean().item() for target in TARGET_COEFFS]
model_stds = [results[target]['estimated_coeffs'].std().item() for target in TARGET_COEFFS]

data_means = [results[target]['conditioning_coeffs'].mean().item() for target in TARGET_COEFFS]

ax_summary.errorbar(TARGET_COEFFS, model_means, yerr=model_stds, fmt='o-', color='#dd8452', capsize=4, label='Generated curves')
ax_summary.plot(TARGET_COEFFS, data_means, 's--', color='#4c72b0', label='Conditioning data')
ax_summary.plot([0.15, 0.85], [0.15, 0.85], linestyle=':', color='black', linewidth=1.5, label='Identity')
ax_summary.set_xlim(0.15, 0.85)
ax_summary.set_ylim(0.1, 0.95)
ax_summary.set_xlabel('Target damping coefficient')
ax_summary.set_ylabel('Estimated coefficient')
ax_summary.set_title('Conditioning signal vs. recovered coefficient')
ax_summary.legend(loc='upper left')
ax_summary.text(
    0.04,
    0.04,
    f'checkpoint = {CHECKPOINT_NAME}\nselected after checking UNet checkpoints',
    transform=ax_summary.transAxes,
    ha='left',
    va='bottom',
    fontsize=9,
    bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '0.8'},
)

for plot_index, target in enumerate(TARGET_COEFFS[::2]):
    ax = fig.add_subplot(grid[plot_index, 1:])
    payload = example_payloads[target]
    conditioning_curve = payload['conditioning_curve']
    ax.plot(conditioning_curve[1], conditioning_curve[0], color='black', linewidth=2.0, label='Conditioning curve')
    for sample_index, generated_curve in enumerate(payload['generated_curves']):
        ax.plot(generated_curve[1], generated_curve[0], linewidth=1.2, alpha=0.75, label='Generated sample' if sample_index == 0 else None)
    ax.set_title(f'Examples around target coefficient {target:.1f}')
    ax.set_xlabel('Irregular time')
    ax.set_ylabel('Value')
    if plot_index == 0:
        ax.legend(loc='upper right')

fig.suptitle('Non-uniform conditional generation mostly preserves local shape, but collapses the inferred damping coefficient', y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_PATH, bbox_inches='tight')
FIGURE_PATH


In [ ]:
for target in TARGET_COEFFS:
    payload = results[target]
    print(
        f"target={target:.1f} | data mean={payload['conditioning_coeffs'].mean():.4f} | generated mean={payload['estimated_coeffs'].mean():.4f} | generated std={payload['estimated_coeffs'].std():.4f}"
    )
